In [1]:
# ============================================
# TVOW Episode Builder (Reproducible Start-to-End)
# ============================================
# pip install -U pandas pyarrow s3fs numpy boto3

import os
import re
import json
import numpy as np
import pandas as pd
import pyarrow.dataset as ds

# ---------- CONFIG ----------
S3_BASE = "s3://customer-dc-grai-matter-prod"

PATHS = {
    "enc":  f"{S3_BASE}/grai_matter_fct_encounters",
    "evt":  f"{S3_BASE}/grai_matter_dim_enc_event_records",
    "adt":  f"{S3_BASE}/grai_matter_dim_adt",                 # optional, may be absent or sparse
    "icd":  f"{S3_BASE}/grai_matter_fct_icd_codes",
    "proc": f"{S3_BASE}/grai_matter_fct_procedure_codes",
    "med":  f"{S3_BASE}/grai_matter_fct_medication",
}

# Output (local). Set S3_UPLOAD_PREFIX to upload back to S3
OUT_EPISODE_CORE   = "episode_core.csv"
OUT_EPISODE_DETAIL = "episode_detail.csv"
S3_UPLOAD_PREFIX   = None  # e.g., "s3://customer-dc-grai-matter-prod/exports"


# ---------- IO HELPERS ----------
def read_parquet_dir(path, columns=None):
    """
    Robustly read a parquet directory (where files are named data_*.snappy.parquet).
    """
    dataset = ds.dataset(path, format="parquet")
    table   = dataset.to_table(columns=columns)
    df      = table.to_pandas(types_mapper=pd.ArrowDtype)
    return df

def to_datetime_utc(s):
    s = pd.to_datetime(s, errors="coerce", utc=True)
    return s

def upload_to_s3(local_path, s3_prefix):
    import boto3
    s3 = boto3.client("s3")
    # Split s3://bucket/prefix/file
    assert s3_prefix.startswith("s3://")
    bucket_key = s3_prefix[5:]
    bucket, key_prefix = bucket_key.split("/", 1)
    key = key_prefix.rstrip("/") + "/" + os.path.basename(local_path)
    s3.upload_file(local_path, bucket, key)
    print(f"Uploaded: {local_path} -> s3://{bucket}/{key}")


# ---------- LOAD DATA (minimal columns) ----------
print("Loading encounters...")
enc_cols = [
    "DH_ENCOUNTER_ID","PRIMARY_PATIENT_IDENTIFIER",
    "ENCOUNTER_START_DT","ENCOUNTER_END_DT",
    "DH_ENCOUNTER_SETTING","DH_ENCOUNTER_STATUS",
    "DH_ENCOUNTER_MEDICAL_SERVICE",
    "DH_IS_ED_ENCOUNTER","DH_ED_ACUITY_LEVEL","DH_ED_ACUITY_LEVEL_NUMERIC"
]
enc = read_parquet_dir(PATHS["enc"], columns=enc_cols).rename(columns={
    "DH_ENCOUNTER_ID":"encounter_id",
    "PRIMARY_PATIENT_IDENTIFIER":"patient_id",
    "ENCOUNTER_START_DT":"encounter_start_dt",
    "ENCOUNTER_END_DT":"encounter_end_dt",
    "DH_ENCOUNTER_SETTING":"enc_setting",
    "DH_ENCOUNTER_STATUS":"enc_status",
    "DH_ENCOUNTER_MEDICAL_SERVICE":"enc_med_service",
    "DH_IS_ED_ENCOUNTER":"is_ed"
})
enc["encounter_start_dt"] = to_datetime_utc(enc["encounter_start_dt"])
enc["encounter_end_dt"]   = to_datetime_utc(enc["encounter_end_dt"])
enc["is_ed"] = enc["is_ed"].astype("boolean")

print("Loading events...")
evt_cols = [
    "DH_ENCOUNTER_ID","PRIMARY_PATIENT_IDENTIFIER",
    "EVENT_DISPLAY_NAME","EVENT_TIME"
]
evt = read_parquet_dir(PATHS["evt"], columns=evt_cols).rename(columns={
    "DH_ENCOUNTER_ID":"encounter_id",
    "PRIMARY_PATIENT_IDENTIFIER":"patient_id",
    "EVENT_DISPLAY_NAME":"event_display_name",
    "EVENT_TIME":"event_time"
})
evt["event_time"] = to_datetime_utc(evt["event_time"])

# ADT table is optional; try best-effort
adt = None
try:
    print("Loading ADT...")
    adt_cols = [
        "DH_ENCOUNTER_ID","PRIMARY_PATIENT_IDENTIFIER",
        "EVENT_TYPE","EVENT_STATUS","EFFECTIVE_TIME"
    ]
    adt = read_parquet_dir(PATHS["adt"], columns=adt_cols).rename(columns={
        "DH_ENCOUNTER_ID":"encounter_id",
        "PRIMARY_PATIENT_IDENTIFIER":"patient_id",
        "EVENT_TYPE":"adt_event_type",
        "EVENT_STATUS":"adt_status",
        "EFFECTIVE_TIME":"adt_time"
    })
    adt["adt_time"] = to_datetime_utc(adt["adt_time"])
except Exception as e:
    print(f"ADT not loaded (continue without): {e}")

print("Loading ICD/procedure/meds...")
icd = read_parquet_dir(PATHS["icd"], columns=[
    "PRIMARY_PATIENT_IDENTIFIER","DH_ENCOUNTER_ID","ICD10_CODE","ICD10_DT"
]).rename(columns={
    "PRIMARY_PATIENT_IDENTIFIER":"patient_id",
    "DH_ENCOUNTER_ID":"encounter_id",
    "ICD10_CODE":"icd10_code",
    "ICD10_DT":"icd_dt"
})
icd["icd_dt"] = to_datetime_utc(icd["icd_dt"])

proc = read_parquet_dir(PATHS["proc"], columns=[
    "PRIMARY_PATIENT_IDENTIFIER","DH_ENCOUNTER_ID",
    "PROCEDURE_CODE","PROCEDURE_DT","PROCEDURE_CODE_DESCRIPTION"
]).rename(columns={
    "PRIMARY_PATIENT_IDENTIFIER":"patient_id",
    "DH_ENCOUNTER_ID":"encounter_id",
    "PROCEDURE_CODE":"proc_code",
    "PROCEDURE_DT":"proc_dt",
    "PROCEDURE_CODE_DESCRIPTION":"proc_desc"
})
proc["proc_dt"] = to_datetime_utc(proc["proc_dt"])

med = read_parquet_dir(PATHS["med"], columns=[
    "PRIMARY_PATIENT_IDENTIFIER","DH_ENCOUNTER_ID",
    "DRUG_CLASS","DRUG_SUBCLASS","DRUG_PRODUCT","ORDER_DT","ORDER_END_DT"
]).rename(columns={
    "PRIMARY_PATIENT_IDENTIFIER":"patient_id",
    "DH_ENCOUNTER_ID":"encounter_id",
    "ORDER_DT":"order_dt",
    "ORDER_END_DT":"order_end_dt"
})
med["order_dt"]     = to_datetime_utc(med["order_dt"])
med["order_end_dt"] = to_datetime_utc(med["order_end_dt"])

print("Shapes:",
      "\n enc:", enc.shape,
      "\n evt:", evt.shape,
      "\n adt:", None if adt is None else adt.shape,
      "\n icd:", icd.shape,
      "\n proc:", proc.shape,
      "\n med:", med.shape)


# ---------- BUILD ADT JOURNEY (department/event heuristics) ----------
# Map event names → department buckets (adjust/extend as you learn local vocab)
DEPT_KEYWORDS = [
    ("Emergency",   ["emergency","ed triage","er", "trauma"]),
    ("Preop",       ["preop","pre-op"]),
    ("OR",          ["operating room","surgery","or in","or start"]),
    ("PACU",        ["pacu","recovery"]),
    ("ICU",         ["icu","intensive care"]),
    ("Med/Surg",    ["med/surg","medical surgical","med surg","nursing - medical / surgical"]),
    ("Lab",         ["laboratory","lab draw","phlebotomy"]),
    ("Radiology",   ["radiology","ct","mri","x-ray","xray","ultrasound"]),
    ("Other",       [])  # default
]

def classify_dept(text: str) -> str:
    if pd.isna(text):
        return "UNK"
    t = str(text).lower()
    for dept, kws in DEPT_KEYWORDS:
        if any(kw in t for kw in kws):
            return dept
    return "Other"

# From enc_event_records
jour_a = (
    evt
      .dropna(subset=["event_time"])
      .assign(event_type="other",  # not explicit in this table
              time_in=lambda d: d["event_time"],
              dept=lambda d: d["event_display_name"].map(classify_dept))
      [["patient_id","encounter_id","dept","event_type","time_in","event_display_name"]]
)

# From ADT (if present): admission/discharge are explicit; dept often not present → leave "UNK"
jour_b = pd.DataFrame()
if adt is not None and not adt.empty:
    jour_b = (
        adt.dropna(subset=["adt_time"])
           .assign(event_type=lambda d: d["adt_event_type"].str.lower(),  # 'Admission','Discharge','Census', etc.
                   time_in=lambda d: d["adt_time"],
                   dept="UNK",
                   event_display_name=lambda d: d["adt_event_type"])
           [["patient_id","encounter_id","dept","event_type","time_in","event_display_name"]]
    )

# Combine, deduplicate, order
journey = (
    pd.concat([jour_a, jour_b], ignore_index=True)
      .drop_duplicates(subset=["patient_id","encounter_id","event_type","time_in","event_display_name"])
)
journey = journey.sort_values(["patient_id","encounter_id","time_in"])

# Compute time_out & durations within encounter
journey["time_out"] = (
    journey
      .groupby(["patient_id","encounter_id"])["time_in"]
      .shift(-1)
)
# If last event lacks time_out, fill from encounter_end_dt
journey = journey.merge(
    enc[["patient_id","encounter_id","encounter_end_dt"]],
    on=["patient_id","encounter_id"], how="left"
)
journey["time_out"] = journey["time_out"].fillna(journey["encounter_end_dt"])
journey["duration_hours"] = (journey["time_out"] - journey["time_in"]).dt.total_seconds()/3600.0

# Event order within episode
journey["event_order"] = (
    journey.groupby(["patient_id","encounter_id"]).cumcount() + 1
)

# Tidy detail table (episode_detail)
episode_detail = (
    journey.rename(columns={"encounter_id":"episode_id",
                            "event_display_name":"EVENT_DISPLAY_NAME"})
           [["episode_id","patient_id","dept","event_type","time_in","time_out",
             "duration_hours","event_order","EVENT_DISPLAY_NAME"]]
)


# ---------- PER-EPISODE SUMMARY ----------
# LOS from encounter header (hours)
enc["los_hours_hdr"] = (enc["encounter_end_dt"] - enc["encounter_start_dt"]).dt.total_seconds()/3600.0

def summarize_episode(g: pd.DataFrame) -> pd.Series:
    dept_series = g["dept"].fillna("UNK").astype(str)
    n_transfers = int(max(((dept_series.shift() != dept_series).sum() - 1), 0))
    return pd.Series({
        "adt_first_time":   g["time_in"].min(),
        "adt_last_time":    g["time_in"].max(),
        "adt_first_dept":   dept_series.iloc[0] if len(dept_series) else np.nan,
        "adt_last_dept":    dept_series.iloc[-1] if len(dept_series) else np.nan,
        "n_events":         int(len(g)),
        "n_unique_events":  int(g["EVENT_DISPLAY_NAME"].nunique()),
        "n_unique_depts":   int(dept_series.nunique()),
        "n_transfers":      n_transfers,
        "adt_dur_hours_sum": float(g["duration_hours"].dropna().sum())
    })

adt_summary = (
    episode_detail
      .groupby(["patient_id","episode_id"], as_index=False)
      .apply(summarize_episode)
)

# ---------- Enrich with ICD/Proc/Med counts ----------
def icd_prefix3(c):
    if pd.isna(c): return np.nan
    c = str(c).strip().upper()
    m = re.match(r"([A-Z]\d{0,2})", c)  # A00, E11, etc.
    return m.group(1) if m else c[:3]

icd["icd3"] = icd["icd10_code"].map(icd_prefix3)

icd_agg = (
    icd.groupby(["patient_id","encounter_id"], as_index=False)
       .agg(
           n_icd=("icd10_code","nunique"),
           n_icd3=("icd3","nunique"),
           icd3_unique=("icd3", lambda s: ",".join(sorted(set([x for x in s.dropna().astype(str)]))[:50]))
       )
       .rename(columns={"encounter_id":"episode_id"})
)

proc_agg = (
    proc.groupby(["patient_id","encounter_id"], as_index=False)
        .agg(n_procs=("proc_code","nunique"))
        .rename(columns={"encounter_id":"episode_id"})
)

med_agg = (
    med.groupby(["patient_id","encounter_id"], as_index=False)
       .agg(
           n_drug_classes=("DRUG_CLASS","nunique"),
           n_drug_subclasses=("DRUG_SUBCLASS","nunique")
       )
       .rename(columns={"encounter_id":"episode_id"})
)

# ---------- Assemble EPISODE CORE ----------
episode_core = (
    enc.rename(columns={"encounter_id":"episode_id"})
       .merge(adt_summary, on=["patient_id","episode_id"], how="left")
       .merge(icd_agg,      on=["patient_id","episode_id"], how="left")
       .merge(proc_agg,     on=["patient_id","episode_id"], how="left")
       .merge(med_agg,      on=["patient_id","episode_id"], how="left")
       .sort_values(["patient_id","episode_id"])
)

# Friendly nulls
for col in ["n_events","n_unique_events","n_unique_depts","n_transfers",
            "adt_dur_hours_sum","n_icd","n_icd3","n_procs","n_drug_classes","n_drug_subclasses"]:
    if col in episode_core:
        episode_core[col] = episode_core[col].fillna(0).astype(np.int64 if "n_" in col else float)

# Sanity checks
assert episode_core["episode_id"].is_unique, "episode_id should be unique in episode_core"
print("Built:",
      "\n episode_detail rows:", len(episode_detail),
      "\n episode_core rows:  ", len(episode_core))

# ---------- SAVE (CSV) ----------
episode_core.to_csv(OUT_EPISODE_CORE, index=False, encoding="utf-8")
episode_detail.to_csv(OUT_EPISODE_DETAIL, index=False, encoding="utf-8")
print(f"Saved:\n  {OUT_EPISODE_CORE}\n  {OUT_EPISODE_DETAIL}")

# ---------- OPTIONAL: UPLOAD BACK TO S3 ----------
if S3_UPLOAD_PREFIX:
    try:
        upload_to_s3(OUT_EPISODE_CORE, S3_UPLOAD_PREFIX)
        upload_to_s3(OUT_EPISODE_DETAIL, S3_UPLOAD_PREFIX)
    except Exception as e:
        print("S3 upload skipped/failed:", e)


Loading encounters...
Loading events...
Loading ADT...
Loading ICD/procedure/meds...
Shapes: 
 enc: (373942, 10) 
 evt: (1290748, 4) 
 adt: (337422, 5) 
 icd: (1151870, 4) 
 proc: (422869, 5) 
 med: (403041, 7)


/tmp/ipykernel_196/2401722549.py:258: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_episode)


Built: 
 episode_detail rows: 1384095 
 episode_core rows:   373942
Saved:
  episode_core.csv
  episode_detail.csv


In [2]:
import pandas as pd
import numpy as np

# --- 1. Load your episode_core file ---
episodes = pd.read_csv("episode_core.csv")

# --- 2. Clean invalid values ---
# (1) Replace negative or unrealistically small durations with NaN
episodes.loc[episodes['adt_dur_hours_sum'] < 0, 'adt_dur_hours_sum'] = np.nan
episodes.loc[episodes['los_hours_hdr'] < 0, 'los_hours_hdr'] = np.nan

# (2) Remove episodes missing encounter start/end
episodes = episodes.dropna(subset=['encounter_start_dt', 'encounter_end_dt'])

# (3) Optionally filter out empty / cancelled encounters
episodes = episodes[episodes['enc_status'].notna() & (episodes['enc_status'] != 'Canceled')]

# (4) Fill minor categorical NaNs with 'Unknown'
cat_cols = ['enc_setting', 'enc_med_service', 'adt_first_dept', 'adt_last_dept']
episodes[cat_cols] = episodes[cat_cols].fillna('Unknown')

# --- 3. Add a few derived metrics ---
episodes['LOS_category'] = pd.cut(
    episodes['los_hours_hdr'],
    bins=[0, 24, 72, np.inf],
    labels=['Short', 'Medium', 'Long']
)
episodes['ICD_diversity'] = episodes['n_icd3'] / (episodes['los_hours_hdr'] + 1)
episodes['Medication_complexity'] = episodes['n_drug_classes'] / (episodes['n_icd3'] + 1)

# --- 4. Save the clean file ---
clean_path = "episode_core_clean.csv"
episodes.to_csv(clean_path, index=False)
print(f"✅ Cleaned dataset saved to: {clean_path}")

✅ Cleaned dataset saved to: episode_core_clean.csv
